This file is for deciding which image size to choose 

I will be trying 224, 128, 160, 256, 320 image sizes

I will be using the targeted augmentation to fix 35/36 class confusion all over this experiment because that was significant confusion

In [17]:
import pandas as pd
import regex as re
import torch

In [18]:
df = pd.read_csv(r'D:\Traffic\labels_processed.csv')

In [19]:
def label_function(dpath):
    class_name = re.findall(r'(\d+)_.*\.png$', dpath.name)
    class_id = int(class_name[0])
    return class_id

In [20]:
from pathlib import Path

In [21]:
path = Path(r'D:\Traffic\traffic_Data_processed\DATA')

In [22]:
lr_head = 0.0017378008365631102
lr_whole_model = 1.9054607491852948e-06

In [23]:
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [24]:
from torch.utils.data import Dataset
from PIL import Image

In [25]:
class dset(Dataset):
    def __init__(self, image_paths, transform = None):
        self.image_paths = image_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        image = Image.open(image_path).convert("RGB")
        label = label_function(Path(image_path))
        if self.transform:
            image = self.transform(image)
        return image, label

In [26]:
image_paths = list(path.rglob("*.png"))

In [27]:
from torchvision import transforms

In [28]:
from torch.utils.data import random_split

In [29]:
from torch.utils.data import DataLoader

In [30]:
import torchvision
num_classes = 55
device = torch.device("cuda")

In [31]:
import kornia.augmentation as K
import torch.nn as nn

In [32]:
transform_1 = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

dataset_1 = dset(image_paths = image_paths, transform = transform_1)

ts = int(0.75*len(dataset_1))
vs = len(dataset_1) - ts
generator = torch.Generator().manual_seed(42)
train_dataset, valid_dataset = random_split(dataset_1, [ts, vs], generator)

train_loader_1 = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

valid_loader_1 = DataLoader(
    valid_dataset,
    batch_size=16,
    shuffle=False
)

model_1 = torchvision.models.resnet34(weights="DEFAULT")
model_1.fc = nn.Linear(
    model_1.fc.in_features,
    num_classes
)

model_1 = model_1.to(device)

for param in model_1.parameters():
    param.requires_grad = False

for param in model_1.fc.parameters():
    param.requires_grad = True

train_aug_1 = K.AugmentationSequential(
    K.ColorJiggle(
        contrast = 0.2,
        p = 0.5
    ),
    K.RandomPlanckianJitter(
        mode = "CIED",
        p = 0.5
    )
).to(device)

targeted_aug_1 = K.AugmentationSequential(
    K.RandomRotation(
        degrees = 10,
        p = 0.5
    ),
    K.RandomAffine(
        degrees = 0,
        scale = (0.9, 1.1),
        p = 0.5
    ),
    K.RandomPerspective(
        distortion_scale = 0.2,
        p = 0.5
    ),
    K.ColorJiggle(
        brightness = 0.2,
        contrast = 0.2,
        p = 0.5
    )
).to(device)

optimizer_1 = torch.optim.RMSprop(
    model_1.fc.parameters(),
    lr=lr_head,
    weight_decay=1e-3
)

scheduler_1 = torch.optim.lr_scheduler.OneCycleLR(
    optimizer_1,
    max_lr = lr_head,
    epochs = 20,
    steps_per_epoch = len(train_loader_1)
)

criterion = nn.CrossEntropyLoss()

for epoch in range(20):
    model_1.train()
    train_loss = 0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader_1:

        images = images.to(device)
        labels = labels.to(device)

        images = train_aug_1(images)

        mask = (labels == 35) | (labels == 36)

        if mask.any():
            images[mask] = targeted_aug_1(images[mask])

        optimizer_1.zero_grad()

        outputs = model_1(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer_1.step()

        scheduler_1.step()

        train_loss += loss.item()

        _, predicted = outputs.max(1)

        train_correct += (predicted == labels).sum().item()

        train_total += labels.size(0)

    train_acc = 100 * train_correct / train_total

    model_1.eval()

    valid_loss = 0
    valid_correct = 0
    valid_total = 0

    with torch.no_grad():

        for images, labels in valid_loader_1:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model_1(images)

            loss = criterion(outputs, labels)

            valid_loss += loss.item()

            _, predicted = outputs.max(1)

            valid_correct += (predicted == labels).sum().item()

            valid_total += labels.size(0)

    valid_acc = 100 * valid_correct / valid_total

    print(
        f"Epoch {epoch+1}/20 | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Valid Acc: {valid_acc:.2f}%"
    )

Epoch 1/20 | Train Acc: 63.57% | Valid Acc: 87.67%
Epoch 2/20 | Train Acc: 88.50% | Valid Acc: 87.19%
Epoch 3/20 | Train Acc: 87.70% | Valid Acc: 91.33%
Epoch 4/20 | Train Acc: 86.77% | Valid Acc: 88.63%
Epoch 5/20 | Train Acc: 88.53% | Valid Acc: 88.92%
Epoch 6/20 | Train Acc: 88.40% | Valid Acc: 93.55%
Epoch 7/20 | Train Acc: 89.17% | Valid Acc: 92.10%
Epoch 8/20 | Train Acc: 90.17% | Valid Acc: 89.98%
Epoch 9/20 | Train Acc: 90.14% | Valid Acc: 91.71%
Epoch 10/20 | Train Acc: 90.27% | Valid Acc: 93.26%
Epoch 11/20 | Train Acc: 91.55% | Valid Acc: 94.12%
Epoch 12/20 | Train Acc: 92.80% | Valid Acc: 95.28%
Epoch 13/20 | Train Acc: 92.77% | Valid Acc: 92.58%
Epoch 14/20 | Train Acc: 94.96% | Valid Acc: 94.80%
Epoch 15/20 | Train Acc: 94.31% | Valid Acc: 96.63%
Epoch 16/20 | Train Acc: 95.66% | Valid Acc: 96.72%
Epoch 17/20 | Train Acc: 97.98% | Valid Acc: 98.17%
Epoch 18/20 | Train Acc: 98.27% | Valid Acc: 98.17%
Epoch 19/20 | Train Acc: 99.20% | Valid Acc: 98.55%
Epoch 20/20 | Train A

In [33]:
transform_2 = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])

dataset_2 = dset(image_paths = image_paths, transform = transform_2)

ts = int(0.75*len(dataset_2))
vs = len(dataset_2) - ts
generator = torch.Generator().manual_seed(42)
train_dataset, valid_dataset = random_split(dataset_2, [ts, vs], generator)

train_loader_2 = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

valid_loader_2 = DataLoader(
    valid_dataset,
    batch_size=16,
    shuffle=False
)

model_2 = torchvision.models.resnet34(weights="DEFAULT")
model_2.fc = nn.Linear(
    model_2.fc.in_features,
    num_classes
)

model_2 = model_2.to(device)

for param in model_2.parameters():
    param.requires_grad = False

for param in model_2.fc.parameters():
    param.requires_grad = True

train_aug_2 = K.AugmentationSequential(
    K.ColorJiggle(
        contrast = 0.2,
        p = 0.5
    ),
    K.RandomPlanckianJitter(
        mode = "CIED",
        p = 0.5
    )
).to(device)

targeted_aug_2 = K.AugmentationSequential(
    K.RandomRotation(
        degrees = 10,
        p = 0.5
    ),
    K.RandomAffine(
        degrees = 0,
        scale = (0.9, 1.1),
        p = 0.5
    ),
    K.RandomPerspective(
        distortion_scale = 0.2,
        p = 0.5
    ),
    K.ColorJiggle(
        brightness = 0.2,
        contrast = 0.2,
        p = 0.5
    )
).to(device)

optimizer_2 = torch.optim.RMSprop(
    model_2.fc.parameters(),
    lr=lr_head,
    weight_decay=1e-3
)

scheduler_2 = torch.optim.lr_scheduler.OneCycleLR(
    optimizer_2,
    max_lr = lr_head,
    epochs = 20,
    steps_per_epoch = len(train_loader_2)
)

criterion = nn.CrossEntropyLoss()

for epoch in range(20):
    model_2.train()
    train_loss = 0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader_2:

        images = images.to(device)
        labels = labels.to(device)

        images = train_aug_2(images)

        mask = (labels == 35) | (labels == 36)

        if mask.any():
            images[mask] = targeted_aug_2(images[mask])

        optimizer_2.zero_grad()

        outputs = model_2(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer_2.step()

        scheduler_2.step()

        train_loss += loss.item()

        _, predicted = outputs.max(1)

        train_correct += (predicted == labels).sum().item()

        train_total += labels.size(0)

    train_acc = 100 * train_correct / train_total

    model_2.eval()

    valid_loss = 0
    valid_correct = 0
    valid_total = 0

    with torch.no_grad():

        for images, labels in valid_loader_2:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model_2(images)

            loss = criterion(outputs, labels)

            valid_loss += loss.item()

            _, predicted = outputs.max(1)

            valid_correct += (predicted == labels).sum().item()

            valid_total += labels.size(0)

    valid_acc = 100 * valid_correct / valid_total

    print(
        f"Epoch {epoch+1}/20 | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Valid Acc: {valid_acc:.2f}%"
    )

Epoch 1/20 | Train Acc: 67.56% | Valid Acc: 88.82%
Epoch 2/20 | Train Acc: 86.22% | Valid Acc: 91.23%
Epoch 3/20 | Train Acc: 86.38% | Valid Acc: 87.57%
Epoch 4/20 | Train Acc: 87.86% | Valid Acc: 88.54%
Epoch 5/20 | Train Acc: 87.41% | Valid Acc: 92.68%
Epoch 6/20 | Train Acc: 86.70% | Valid Acc: 93.06%
Epoch 7/20 | Train Acc: 88.27% | Valid Acc: 88.44%
Epoch 8/20 | Train Acc: 91.07% | Valid Acc: 92.49%
Epoch 9/20 | Train Acc: 88.89% | Valid Acc: 93.74%
Epoch 10/20 | Train Acc: 90.46% | Valid Acc: 93.64%
Epoch 11/20 | Train Acc: 91.20% | Valid Acc: 94.89%
Epoch 12/20 | Train Acc: 91.94% | Valid Acc: 93.06%
Epoch 13/20 | Train Acc: 91.97% | Valid Acc: 94.89%
Epoch 14/20 | Train Acc: 93.09% | Valid Acc: 95.38%
Epoch 15/20 | Train Acc: 93.90% | Valid Acc: 95.09%
Epoch 16/20 | Train Acc: 95.15% | Valid Acc: 96.15%
Epoch 17/20 | Train Acc: 96.88% | Valid Acc: 97.59%
Epoch 18/20 | Train Acc: 97.33% | Valid Acc: 97.11%
Epoch 19/20 | Train Acc: 98.43% | Valid Acc: 97.69%
Epoch 20/20 | Train A

In [34]:
transform_3 = transforms.Compose([
    transforms.Resize((160, 160)),
    transforms.ToTensor(),
])

dataset_3 = dset(image_paths = image_paths, transform = transform_3)

ts = int(0.75*len(dataset_3))
vs = len(dataset_3) - ts
generator = torch.Generator().manual_seed(42)
train_dataset, valid_dataset = random_split(dataset_3, [ts, vs], generator)

train_loader_3 = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

valid_loader_3 = DataLoader(
    valid_dataset,
    batch_size=16,
    shuffle=False
)

model_3 = torchvision.models.resnet34(weights="DEFAULT")
model_3.fc = nn.Linear(
    model_3.fc.in_features,
    num_classes
)

model_3 = model_3.to(device)

for param in model_3.parameters():
    param.requires_grad = False

for param in model_3.fc.parameters():
    param.requires_grad = True

train_aug_3 = K.AugmentationSequential(
    K.ColorJiggle(
        contrast = 0.2,
        p = 0.5
    ),
    K.RandomPlanckianJitter(
        mode = "CIED",
        p = 0.5
    )
).to(device)

targeted_aug_3 = K.AugmentationSequential(
    K.RandomRotation(
        degrees = 10,
        p = 0.5
    ),
    K.RandomAffine(
        degrees = 0,
        scale = (0.9, 1.1),
        p = 0.5
    ),
    K.RandomPerspective(
        distortion_scale = 0.2,
        p = 0.5
    ),
    K.ColorJiggle(
        brightness = 0.2,
        contrast = 0.2,
        p = 0.5
    )
).to(device)

optimizer_3 = torch.optim.RMSprop(
    model_3.fc.parameters(),
    lr=lr_head,
    weight_decay=1e-3
)

scheduler_3 = torch.optim.lr_scheduler.OneCycleLR(
    optimizer_3,
    max_lr = lr_head,
    epochs = 20,
    steps_per_epoch = len(train_loader_3)
)

criterion = nn.CrossEntropyLoss()

for epoch in range(20):
    model_3.train()
    train_loss = 0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader_3:

        images = images.to(device)
        labels = labels.to(device)

        images = train_aug_3(images)

        mask = (labels == 35) | (labels == 36)

        if mask.any():
            images[mask] = targeted_aug_3(images[mask])

        optimizer_3.zero_grad()

        outputs = model_3(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer_3.step()

        scheduler_3.step()

        train_loss += loss.item()

        _, predicted = outputs.max(1)

        train_correct += (predicted == labels).sum().item()

        train_total += labels.size(0)

    train_acc = 100 * train_correct / train_total

    model_3.eval()

    valid_loss = 0
    valid_correct = 0
    valid_total = 0

    with torch.no_grad():

        for images, labels in valid_loader_3:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model_3(images)

            loss = criterion(outputs, labels)

            valid_loss += loss.item()

            _, predicted = outputs.max(1)

            valid_correct += (predicted == labels).sum().item()

            valid_total += labels.size(0)

    valid_acc = 100 * valid_correct / valid_total

    print(
        f"Epoch {epoch+1}/20 | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Valid Acc: {valid_acc:.2f}%"
    )

Epoch 1/20 | Train Acc: 68.42% | Valid Acc: 88.25%
Epoch 2/20 | Train Acc: 88.79% | Valid Acc: 91.04%
Epoch 3/20 | Train Acc: 85.61% | Valid Acc: 83.62%
Epoch 4/20 | Train Acc: 87.25% | Valid Acc: 91.04%
Epoch 5/20 | Train Acc: 88.82% | Valid Acc: 91.52%
Epoch 6/20 | Train Acc: 89.53% | Valid Acc: 89.50%
Epoch 7/20 | Train Acc: 88.72% | Valid Acc: 92.00%
Epoch 8/20 | Train Acc: 90.59% | Valid Acc: 88.44%
Epoch 9/20 | Train Acc: 90.56% | Valid Acc: 88.44%
Epoch 10/20 | Train Acc: 90.81% | Valid Acc: 92.68%
Epoch 11/20 | Train Acc: 91.78% | Valid Acc: 93.64%
Epoch 12/20 | Train Acc: 92.55% | Valid Acc: 94.32%
Epoch 13/20 | Train Acc: 92.03% | Valid Acc: 94.80%
Epoch 14/20 | Train Acc: 94.22% | Valid Acc: 94.61%
Epoch 15/20 | Train Acc: 95.09% | Valid Acc: 95.47%
Epoch 16/20 | Train Acc: 95.28% | Valid Acc: 96.24%
Epoch 17/20 | Train Acc: 96.72% | Valid Acc: 96.63%
Epoch 18/20 | Train Acc: 97.21% | Valid Acc: 97.30%
Epoch 19/20 | Train Acc: 99.13% | Valid Acc: 97.78%
Epoch 20/20 | Train A

In [35]:
transform_4 = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
])

dataset_4 = dset(image_paths = image_paths, transform = transform_4)

ts = int(0.75*len(dataset_4))
vs = len(dataset_4) - ts
generator = torch.Generator().manual_seed(42)
train_dataset, valid_dataset = random_split(dataset_4, [ts, vs], generator)

train_loader_4 = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

valid_loader_4 = DataLoader(
    valid_dataset,
    batch_size=16,
    shuffle=False
)

model_4 = torchvision.models.resnet34(weights="DEFAULT")
model_4.fc = nn.Linear(
    model_4.fc.in_features,
    num_classes
)

model_4 = model_4.to(device)

for param in model_4.parameters():
    param.requires_grad = False

for param in model_4.fc.parameters():
    param.requires_grad = True

train_aug_4 = K.AugmentationSequential(
    K.ColorJiggle(
        contrast = 0.2,
        p = 0.5
    ),
    K.RandomPlanckianJitter(
        mode = "CIED",
        p = 0.5
    )
).to(device)

targeted_aug_4 = K.AugmentationSequential(
    K.RandomRotation(
        degrees = 10,
        p = 0.5
    ),
    K.RandomAffine(
        degrees = 0,
        scale = (0.9, 1.1),
        p = 0.5
    ),
    K.RandomPerspective(
        distortion_scale = 0.2,
        p = 0.5
    ),
    K.ColorJiggle(
        brightness = 0.2,
        contrast = 0.2,
        p = 0.5
    )
).to(device)

optimizer_4 = torch.optim.RMSprop(
    model_4.fc.parameters(),
    lr=lr_head,
    weight_decay=1e-3
)

scheduler_4 = torch.optim.lr_scheduler.OneCycleLR(
    optimizer_4,
    max_lr = lr_head,
    epochs = 20,
    steps_per_epoch = len(train_loader_4)
)

criterion = nn.CrossEntropyLoss()

for epoch in range(20):
    model_4.train()
    train_loss = 0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader_4:

        images = images.to(device)
        labels = labels.to(device)

        images = train_aug_4(images)

        mask = (labels == 35) | (labels == 36)

        if mask.any():
            images[mask] = targeted_aug_4(images[mask])

        optimizer_4.zero_grad()

        outputs = model_4(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer_4.step()

        scheduler_4.step()

        train_loss += loss.item()

        _, predicted = outputs.max(1)

        train_correct += (predicted == labels).sum().item()

        train_total += labels.size(0)

    train_acc = 100 * train_correct / train_total

    model_4.eval()

    valid_loss = 0
    valid_correct = 0
    valid_total = 0

    with torch.no_grad():

        for images, labels in valid_loader_4:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model_4(images)

            loss = criterion(outputs, labels)

            valid_loss += loss.item()

            _, predicted = outputs.max(1)

            valid_correct += (predicted == labels).sum().item()

            valid_total += labels.size(0)

    valid_acc = 100 * valid_correct / valid_total

    print(
        f"Epoch {epoch+1}/20 | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Valid Acc: {valid_acc:.2f}%"
    )

Epoch 1/20 | Train Acc: 62.51% | Valid Acc: 80.92%
Epoch 2/20 | Train Acc: 86.93% | Valid Acc: 84.49%
Epoch 3/20 | Train Acc: 86.48% | Valid Acc: 85.07%
Epoch 4/20 | Train Acc: 86.41% | Valid Acc: 85.07%
Epoch 5/20 | Train Acc: 87.54% | Valid Acc: 89.88%
Epoch 6/20 | Train Acc: 88.95% | Valid Acc: 92.87%
Epoch 7/20 | Train Acc: 88.05% | Valid Acc: 94.03%
Epoch 8/20 | Train Acc: 88.89% | Valid Acc: 89.40%
Epoch 9/20 | Train Acc: 90.33% | Valid Acc: 92.49%
Epoch 10/20 | Train Acc: 89.37% | Valid Acc: 94.70%
Epoch 11/20 | Train Acc: 93.38% | Valid Acc: 94.03%
Epoch 12/20 | Train Acc: 92.42% | Valid Acc: 92.87%
Epoch 13/20 | Train Acc: 93.58% | Valid Acc: 94.32%
Epoch 14/20 | Train Acc: 91.94% | Valid Acc: 94.89%
Epoch 15/20 | Train Acc: 94.47% | Valid Acc: 95.95%
Epoch 16/20 | Train Acc: 96.08% | Valid Acc: 95.95%
Epoch 17/20 | Train Acc: 96.21% | Valid Acc: 97.30%
Epoch 18/20 | Train Acc: 98.62% | Valid Acc: 97.69%
Epoch 19/20 | Train Acc: 99.07% | Valid Acc: 97.88%
Epoch 20/20 | Train A

In [36]:
transform_5 = transforms.Compose([
    transforms.Resize((320, 320)),
    transforms.ToTensor(),
])

dataset_5 = dset(image_paths = image_paths, transform = transform_5)

ts = int(0.75*len(dataset_5))
vs = len(dataset_5) - ts
generator = torch.Generator().manual_seed(42)
train_dataset, valid_dataset = random_split(dataset_5, [ts, vs], generator)

train_loader_5 = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

valid_loader_5 = DataLoader(
    valid_dataset,
    batch_size=16,
    shuffle=False
)

model_5 = torchvision.models.resnet34(weights="DEFAULT")
model_5.fc = nn.Linear(
    model_5.fc.in_features,
    num_classes
)

model_5 = model_5.to(device)

for param in model_5.parameters():
    param.requires_grad = False

for param in model_5.fc.parameters():
    param.requires_grad = True

train_aug_5 = K.AugmentationSequential(
    K.ColorJiggle(
        contrast = 0.2,
        p = 0.5
    ),
    K.RandomPlanckianJitter(
        mode = "CIED",
        p = 0.5
    )
).to(device)

targeted_aug_5 = K.AugmentationSequential(
    K.RandomRotation(
        degrees = 10,
        p = 0.5
    ),
    K.RandomAffine(
        degrees = 0,
        scale = (0.9, 1.1),
        p = 0.5
    ),
    K.RandomPerspective(
        distortion_scale = 0.2,
        p = 0.5
    ),
    K.ColorJiggle(
        brightness = 0.2,
        contrast = 0.2,
        p = 0.5
    )
).to(device)

optimizer_5 = torch.optim.RMSprop(
    model_5.fc.parameters(),
    lr=lr_head,
    weight_decay=1e-3
)

scheduler_5 = torch.optim.lr_scheduler.OneCycleLR(
    optimizer_5,
    max_lr = lr_head,
    epochs = 20,
    steps_per_epoch = len(train_loader_5)
)

criterion = nn.CrossEntropyLoss()

for epoch in range(20):
    model_5.train()
    train_loss = 0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader_5:

        images = images.to(device)
        labels = labels.to(device)

        images = train_aug_5(images)

        mask = (labels == 35) | (labels == 36)

        if mask.any():
            images[mask] = targeted_aug_5(images[mask])

        optimizer_5.zero_grad()

        outputs = model_5(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer_5.step()

        scheduler_5.step()

        train_loss += loss.item()

        _, predicted = outputs.max(1)

        train_correct += (predicted == labels).sum().item()

        train_total += labels.size(0)

    train_acc = 100 * train_correct / train_total

    model_5.eval()

    valid_loss = 0
    valid_correct = 0
    valid_total = 0

    with torch.no_grad():

        for images, labels in valid_loader_5:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model_5(images)

            loss = criterion(outputs, labels)

            valid_loss += loss.item()

            _, predicted = outputs.max(1)

            valid_correct += (predicted == labels).sum().item()

            valid_total += labels.size(0)

    valid_acc = 100 * valid_correct / valid_total

    print(
        f"Epoch {epoch+1}/20 | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Valid Acc: {valid_acc:.2f}%"
    )

Epoch 1/20 | Train Acc: 56.12% | Valid Acc: 87.57%
Epoch 2/20 | Train Acc: 83.46% | Valid Acc: 84.20%
Epoch 3/20 | Train Acc: 85.13% | Valid Acc: 88.34%
Epoch 4/20 | Train Acc: 83.36% | Valid Acc: 84.59%
Epoch 5/20 | Train Acc: 86.09% | Valid Acc: 87.76%
Epoch 6/20 | Train Acc: 87.25% | Valid Acc: 90.66%
Epoch 7/20 | Train Acc: 88.08% | Valid Acc: 88.15%
Epoch 8/20 | Train Acc: 88.11% | Valid Acc: 88.63%
Epoch 9/20 | Train Acc: 89.40% | Valid Acc: 87.86%
Epoch 10/20 | Train Acc: 89.40% | Valid Acc: 92.20%
Epoch 11/20 | Train Acc: 91.23% | Valid Acc: 93.83%
Epoch 12/20 | Train Acc: 91.26% | Valid Acc: 89.79%
Epoch 13/20 | Train Acc: 92.03% | Valid Acc: 91.91%
Epoch 14/20 | Train Acc: 92.58% | Valid Acc: 94.41%
Epoch 15/20 | Train Acc: 94.31% | Valid Acc: 91.52%
Epoch 16/20 | Train Acc: 95.57% | Valid Acc: 96.24%
Epoch 17/20 | Train Acc: 97.14% | Valid Acc: 97.69%
Epoch 18/20 | Train Acc: 98.20% | Valid Acc: 97.21%
Epoch 19/20 | Train Acc: 99.10% | Valid Acc: 97.98%
Epoch 20/20 | Train A

Ok so as we can see that most well performing was that model which was having images resized to 224x224 